DNS (Domain Name System) translates human-readable names into IP addresses. Azure provides two DNS services that cover different scenarios:

- **Azure DNS** — host your public DNS zones (the authoritative nameserver for your domain) and private DNS zones (name resolution within VNets)
- **Azure Traffic Manager** — DNS-based global traffic routing across endpoints in different regions or clouds

Together these services let you manage your entire DNS and global routing strategy without running your own DNS infrastructure.

## Azure DNS — Public Zones

**Azure DNS** hosts public DNS zones — it becomes the authoritative nameserver for your domain. When someone queries `www.contoso.com`, Azure DNS answers the query.

Azure DNS does **not** register domains — you register domains through a domain registrar (GoDaddy, Namecheap, Azure App Service Domains, etc.) and then delegate the zone to Azure DNS by updating the registrar's nameserver (NS) records.

### Supported record types

| Type | Description | Example |
|---|---|---|
| **A** | IPv4 address | `www → 203.0.113.10` |
| **AAAA** | IPv6 address | `www → 2001:db8::1` |
| **CNAME** | Canonical name (alias) | `www → myapp.azurewebsites.net` |
| **MX** | Mail exchange | `@ → mail.contoso.com (priority 10)` |
| **TXT** | Text — used for SPF, DKIM, domain verification | `@ → "v=spf1 include:..."` |
| **NS** | Nameserver — delegates a subdomain | `sub → ns1.example.com` |
| **SOA** | Start of authority — auto-created per zone | — |
| **SRV** | Service locator | `_sip._tcp → sip.contoso.com:5060` |
| **PTR** | Reverse DNS — IP to name | `10.0.0.1 → vm1.contoso.com` |
| **CAA** | Certificate authority authorisation | `@ → 0 issue "letsencrypt.org"` |
| **Alias (ANAME)** | Azure-specific — point apex to Azure resource | `@ → myapp.azurewebsites.net` |

### Alias records

A standard DNS limitation is that a **CNAME cannot be used at the zone apex** (the bare domain — `contoso.com` without a subdomain). Azure DNS solves this with **Alias records** — they point the apex directly to an Azure resource (Public IP, Load Balancer, Front Door, Traffic Manager profile) and automatically track IP changes as the resource is updated.

```
contoso.com  →  Alias  →  myapp-pip (Standard Public IP)
                           (IP auto-updated if Public IP changes)
```

### Delegation to Azure DNS

```
1. Create zone in Azure DNS → Azure assigns 4 NS records (e.g. ns1-01.azure-dns.com)
2. Log into your registrar → update NS records to the Azure DNS nameservers
3. DNS propagation: minutes to 48 hours
```

### Zone delegation (subdomains)

Delegate a subdomain to a different DNS zone — useful when a team manages their own sub-zone:

```
contoso.com zone:  dev.contoso.com  NS  ns1-02.azure-dns.com  ← delegation record
dev.contoso.com zone: (separate Azure DNS zone managed by dev team)
```

## Azure DNS — Private Zones

**Azure Private DNS Zones** provide name resolution for resources within one or more VNets — without exposing records publicly or deploying your own DNS servers.

```
Private zone: internal.contoso.com
  vm1.internal.contoso.com  →  10.0.1.4
  db.internal.contoso.com   →  10.0.2.5
```

### VNet links

A **VNet link** associates a private DNS zone with a VNet. Resources in linked VNets can resolve records in the zone. A zone can be linked to multiple VNets — and a VNet can be linked to multiple private zones.

| Link setting | Description |
|---|---|
| **Auto-registration enabled** | VMs in the linked VNet automatically get an A record; deleted when VM is removed |
| **Auto-registration disabled** | Records must be managed manually — useful for static resources |

Auto-registration only creates records for VMs — not for private endpoints, load balancers, or other resources.

### Private DNS zones for private endpoints

Every Azure PaaS service has a dedicated `privatelink.*` DNS zone used when a private endpoint is created:

| Service | Private DNS zone |
|---|---|
| Azure Blob Storage | `privatelink.blob.core.windows.net` |
| Azure Files | `privatelink.file.core.windows.net` |
| Azure SQL Database | `privatelink.database.windows.net` |
| Azure Key Vault | `privatelink.vaultcore.azure.net` |
| Azure Container Registry | `privatelink.azurecr.io` |
| App Service / Functions | `privatelink.azurewebsites.net` |

When a private endpoint is created, Azure can automatically create the `privatelink.*` zone, add an A record for the private IP, and link the zone to your VNet — so the FQDN resolves to the private IP from within the VNet.

### Azure DNS Private Resolver

In hybrid environments where on-premises DNS servers need to resolve Azure private DNS zone names (or vice versa), the **Azure DNS Private Resolver** replaces the need for custom DNS server VMs:

- **Inbound endpoint** — on-premises DNS servers forward queries for Azure private zones to this IP
- **Outbound endpoint + forwarding ruleset** — VNets forward queries for on-premises domains to on-premises DNS servers

```
On-prem DNS  →  forward *.internal.contoso.com  →  Inbound endpoint (10.0.5.4)
                                                    Azure DNS Private Resolver
                                                    resolves from private zone
```

## Azure Traffic Manager

**Azure Traffic Manager** is a DNS-based global traffic router. It does not proxy or inspect traffic — it returns a DNS response directing the client to the best endpoint. The client then connects directly.

```
Client  →  DNS query: myapp.trafficmanager.net
        ←  DNS response: eastus-endpoint.contoso.com  (best endpoint)
Client  →  connects directly to eastus-endpoint.contoso.com
```

### Traffic Manager profile

A **Traffic Manager profile** contains:
- A **DNS name** (`<profile-name>.trafficmanager.net`) — point your domain's CNAME here
- A **routing method** — how endpoints are selected
- **Endpoints** — the destinations Traffic Manager routes to
- **Health monitoring settings** — how endpoints are probed

### Endpoint types

| Type | Description |
|---|---|
| **Azure endpoints** | Azure resources with a public IP or DNS name (App Service, Cloud Service, Public IP) |
| **External endpoints** | Any endpoint with an IPv4/IPv6 address or FQDN — on-premises or other clouds |
| **Nested endpoints** | Another Traffic Manager profile — for combining routing methods |

### Routing methods

| Method | How it works | Use case |
|---|---|---|
| **Performance** | Routes to the endpoint with the lowest latency from the client's location | Active-active multi-region — best UX |
| **Weighted** | Distributes traffic proportionally by weight (1–1000) | Canary deployments, gradual migrations |
| **Priority** | All traffic to highest-priority endpoint; failover to lower priority when unhealthy | Active-passive DR failover |
| **Geographic** | Routes based on the DNS query's geographic origin | Data residency, legal compliance |
| **Multivalue** | Returns multiple healthy endpoints in the DNS response | Redundancy — client picks |
| **Subnet** | Routes based on the client's IP address range | Different experiences for known IP blocks |

### Health monitoring

Traffic Manager probes each endpoint on a configurable interval (10 or 30 seconds). If an endpoint fails a configurable number of consecutive probes, it is marked **Degraded** and excluded from DNS responses until it recovers.

| Setting | Options |
|---|---|
| Protocol | HTTP, HTTPS, TCP |
| Port | Any |
| Path | For HTTP/HTTPS — e.g. `/health` |
| Probe interval | 10 s (fast) or 30 s (normal) |
| Tolerated failures | Number of consecutive failures before Degraded |

### TTL and failover speed

Traffic Manager sets a **DNS TTL** on its responses (default 60 seconds, minimum 0). Clients cache the DNS result for the TTL duration — so failover speed is bounded by the TTL. Lowering the TTL to 0 allows near-instant failover but increases DNS query load.

### Nested profiles

Combine routing methods using nested profiles:

```
Parent profile (Priority routing)
  ├── Child profile A (Performance routing — primary regions)
  └── Child profile B (Weighted routing — DR region)
```

Within the primary regions, performance routing picks the lowest-latency region. If all primary regions fail, traffic falls over to the DR region via the parent priority routing.

## DNS Resolution Flows

### Public resolution with Traffic Manager

```
1. User browses to contoso.com
2. Recursive resolver queries Azure DNS (authoritative for contoso.com)
3. Azure DNS returns CNAME: contoso.trafficmanager.net
4. Recursive resolver queries Traffic Manager
5. Traffic Manager returns A record: 203.0.113.10 (best endpoint, TTL=60s)
6. Browser connects to 203.0.113.10
```

### Private resolution within a VNet

```
1. VM queries 168.63.129.16 (Azure-provided DNS)
2. Azure DNS checks linked private zones
3. Matches db.internal.contoso.com → 10.0.2.5
4. Returns private IP — traffic stays in VNet
```

### Hybrid resolution (Private Resolver)

```
1. On-prem server queries on-prem DNS for vm1.internal.contoso.com
2. On-prem DNS has conditional forwarder: *.internal.contoso.com → 10.0.5.4 (inbound endpoint)
3. Azure DNS Private Resolver receives query, resolves from private zone
4. Returns 10.0.1.4 — private IP of the VM
```

## Custom Domains with Azure Services

### App Service custom domain

```
Option A — subdomain (CNAME):
  www.contoso.com  CNAME  myapp.azurewebsites.net

Option B — apex domain (Alias record in Azure DNS):
  contoso.com  ALIAS  myapp-pip  (Standard Public IP attached to App Service)

Verification:
  asuid.www.contoso.com  TXT  <domain verification ID from App Service>
```

### Front Door custom domain

```
www.contoso.com  CNAME  myfd.azurefd.net
contoso.com      ALIAS  myfd (Azure Front Door profile)  ← apex via Azure DNS alias
```

Front Door can issue and auto-renew a **managed TLS certificate** for the custom domain — no manual certificate management needed.

### Traffic Manager custom domain

```
myapp.contoso.com  CNAME  myprofile.trafficmanager.net
contoso.com        ALIAS  myprofile (Traffic Manager profile)  ← apex
```

## Working with Azure DNS and Traffic Manager via Python

In [ ]:
# pip install azure-identity azure-mgmt-dns azure-mgmt-trafficmanager

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.dns import DnsManagementClient
from azure.mgmt.dns.models import Zone, RecordSet, ARecord, CnameRecord, TxtRecord, MxRecord
from azure.mgmt.trafficmanager import TrafficManagerManagementClient
from azure.mgmt.trafficmanager.models import Profile, DnsConfig, MonitorConfig, Endpoint

credential      = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"

dns = DnsManagementClient(credential, subscription_id)
tm  = TrafficManagerManagementClient(credential, subscription_id)

In [ ]:
# Create a public DNS zone
zone = dns.zones.create_or_update(
    resource_group, "contoso.com",
    Zone(location="global")
)
print(f"Zone: {zone.name}")
print("Nameservers (update at your registrar):")
for ns in zone.name_servers:
    print(f"  {ns}")

In [ ]:
# A record — web server
dns.record_sets.create_or_update(
    resource_group, "contoso.com", "www", "A",
    RecordSet(ttl=3600, a_records=[ARecord(ipv4_address="203.0.113.10")])
)
print("Created: www A 203.0.113.10")

# CNAME record — alias to App Service
dns.record_sets.create_or_update(
    resource_group, "contoso.com", "app", "CNAME",
    RecordSet(ttl=3600, cname_record=CnameRecord(cname="myapp.azurewebsites.net"))
)
print("Created: app CNAME myapp.azurewebsites.net")

# TXT record — SPF
dns.record_sets.create_or_update(
    resource_group, "contoso.com", "@", "TXT",
    RecordSet(ttl=3600, txt_records=[TxtRecord(value=["v=spf1 include:spf.protection.outlook.com -all"])])
)
print("Created: @ TXT SPF record")

# MX record — mail
dns.record_sets.create_or_update(
    resource_group, "contoso.com", "@", "MX",
    RecordSet(ttl=3600, mx_records=[MxRecord(preference=10, exchange="mail.protection.outlook.com")])
)
print("Created: @ MX mail.protection.outlook.com (priority 10)")

In [ ]:
# List all records in a zone
for rs in dns.record_sets.list_by_dns_zone(resource_group, "contoso.com"):
    rtype = rs.type.split("/")[-1]  # e.g. "A", "CNAME"
    print(f"  {rs.name:<30} {rtype:<8} TTL:{rs.ttl}")

In [ ]:
from azure.mgmt.network import NetworkManagementClient
from azure.mgmt.network.models import PrivateDnsZone, VirtualNetworkLink

net = NetworkManagementClient(credential, subscription_id)

# Create a private DNS zone
poller = net.private_zones.begin_create_or_update(
    resource_group, "internal.contoso.com",
    PrivateDnsZone(location="global")
)
private_zone = poller.result()
print(f"Private zone: {private_zone.name}")

# Link the private zone to a VNet with auto-registration
vnet_id = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/virtualNetworks/my-vnet"
poller = net.virtual_network_links.begin_create_or_update(
    resource_group, "internal.contoso.com", "my-vnet-link",
    VirtualNetworkLink(
        location="global",
        virtual_network={"id": vnet_id},
        registration_enabled=True  # auto-register VM hostnames
    )
)
link = poller.result()
print(f"VNet link: {link.name}  auto-registration: {link.registration_enabled}")

In [ ]:
# Create a Traffic Manager profile with Performance routing
profile = tm.profiles.create_or_update(
    resource_group, "my-tm-profile",
    Profile(
        location="global",
        profile_status="Enabled",
        traffic_routing_method="Performance",
        dns_config=DnsConfig(
            relative_name="my-tm-profile",  # → my-tm-profile.trafficmanager.net
            ttl=60
        ),
        monitor_config=MonitorConfig(
            protocol="HTTPS",
            port=443,
            path="/health",
            interval_in_seconds=30,
            timeout_in_seconds=10,
            tolerated_number_of_failures=3
        )
    )
)
print(f"Profile: {profile.name}  DNS: {profile.dns_config.fqdn}")

In [ ]:
# Add two Azure endpoints (App Services in different regions)
eastus_app_id = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Web/sites/myapp-eastus"
westus_app_id = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Web/sites/myapp-westus"

for name, resource_id in [("eastus-ep", eastus_app_id), ("westus-ep", westus_app_id)]:
    ep = tm.endpoints.create_or_update(
        resource_group, "my-tm-profile", "AzureEndpoints", name,
        Endpoint(
            type="Microsoft.Network/trafficManagerProfiles/azureEndpoints",
            target_resource_id=resource_id,
            endpoint_status="Enabled"
        )
    )
    print(f"Endpoint added: {ep.name}  status: {ep.endpoint_status}")

# Check endpoint health
profile = tm.profiles.get(resource_group, "my-tm-profile")
for ep in (profile.endpoints or []):
    print(f"  {ep.name:<20} monitor status: {ep.endpoint_monitor_status}")

## Summary

| Concept | Key Takeaway |
|---|---|
| Azure DNS (public) | Hosts authoritative DNS zones; does not register domains — delegate NS at your registrar |
| Alias record | Azure-specific — points apex domain to Azure resource; tracks IP changes automatically |
| CNAME at apex | Not allowed in standard DNS — use Alias record in Azure DNS instead |
| Zone delegation | NS record in parent zone pointing subdomain to a child zone's nameservers |
| Azure Private DNS Zone | Custom DNS within VNets; link to VNet; auto-registration for VMs |
| privatelink.* zones | Used by private endpoints — FQDN resolves to private IP from within VNet |
| DNS Private Resolver | Enables hybrid DNS — on-prem forwards to Azure private zones and vice versa |
| Traffic Manager | DNS-based global router; does not proxy traffic; works with any protocol |
| TM profile | DNS name + routing method + endpoints + health monitoring |
| Performance routing | Route to lowest-latency endpoint from client location |
| Weighted routing | Distribute by weight — canary deployments and migrations |
| Priority routing | Active-passive failover — all traffic to P1, failover to P2 |
| Geographic routing | Route by client region — data residency compliance |
| Nested profiles | Combine routing methods — e.g. Performance within Priority for DR |
| TTL and failover | Lower TTL = faster failover but higher DNS query volume; default 60 s |
| TM vs Front Door | TM = DNS-only, any protocol; Front Door = L7 proxy, HTTP/HTTPS, CDN, near-instant failover |